In [ ]:
import os
import sys 
os.chdir("/workspaces/dev")
sys.path.append("/workspaces/dev/modules")

In [ ]:
from RTWhisper import TokenStreamer, Hyperparameters
import librosa
import numpy as np

In [ ]:
MODEL_SIZE = "large-v3"

SAMPLE_RATE = 16000
BUFFER_SIZE = 10

In [ ]:
audio, sr = librosa.load("/workspaces/dev/.data/news_with_English.mp3", sr=SAMPLE_RATE)

In [ ]:
# audio = audio[85 * SAMPLE_RATE:]

In [ ]:
total_samples = len(audio)

segments = []
pos = 0
while pos < total_samples:
  rand_len = int(np.random.normal(loc=48000, scale=400))
  rand_len = np.clip(rand_len, 46000, 50000)
  end = min(pos + rand_len, total_samples)
  # end = min(pos + 16000, total_samples)

  chunk = audio[pos:end]
  segments.append(chunk)
  pos = end

In [ ]:
# full_text = ""
# for segment in segments:
#   if len(segment) < 160:
#     continue
#   seg, info = whisper.translate(segment, language="ko")
#   for s in seg:
#     full_text += s.text

In [ ]:
# print(full_text)

In [ ]:
HYPERPARAMETERS = {
  "sentence_max_prev_sentence": 1,
  "weighted_and_offset_token_boundary": 8000,
  "duration_filter_z": {
    "default": 2.0,
    "ko": 2.0,
    "en": 2.0,
  },
  "probability_filter": {
    "z": {
      "default": 2.0,
      "ko": 2.0,
      "en": 2.0,
    },
    "min_prob": {
      "default": 1.0,
      "ko": 0.4,
      "en": 0.4,
    },
  },
  "selector": {
    "search_range_sc": {
      "default": 24000,
      "ko": 24000,
      "en": 24000,
    },
    "threshold": {
      "default": 0.5,
      "ko": 0.25,
      "en": 0.5,
    },
    "padding": {
      "default": 3200,
      "ko": 3200,
      "en": 3200,
    },
    "tolerance": {
      "default": 8000,
      "ko": 8000,
      "en": 8000,
    },
  },
  "classifier_max_prev_sc": {
    "default": 96000
  }
}

In [ ]:
hyper = Hyperparameters(None, HYPERPARAMETERS)

In [ ]:
whisper_service = TokenStreamer.get_instance(hyper)

In [ ]:
raise Exception("stop")

In [ ]:
from RTWhisper.data import Param
from IPython.display import Audio

In [ ]:
segment_id = 0
completed = []
param = Param()

In [ ]:
segment = segments[segment_id]
segment_id += 1

param.audio = segment

result = whisper_service.process(param)
completed.extend(result.completed)

print(f"{segment_id}" + "--" * 20)
print([(v.lang, v.text) for v in completed])
print([(v.lang, v.text) for v in result.completed])
print([(v.lang, v.text) for v in result.candidate])
print([(v.lang, v.text) for v in result.prev_completed_tokens if v.is_word])
print([(v.lang, v.text) for v in result.prev_candidate_tokens if v.is_word])

param.update(result)

Audio(result.prev_processed_audio, rate=SAMPLE_RATE)

In [ ]:
Audio(segment, rate=SAMPLE_RATE)

In [ ]:
for segment in segments:
  param.audio = segment

  result = whisper_service.process(param)
  completed.extend(result.completed)

  print(f"{segment_id}" + "--" * 20)
  # print([(v.lang, v.text) for v in completed])
  print([(v.lang, v.text) for v in result.completed])
  print([(v.lang, v.text) for v in result.candidate])
  # print([(v.lang, v.text) for v in result.prev_completed_tokens if v.is_word])
  # print([(v.lang, v.text) for v in result.prev_candidate_tokens if v.is_word])

  param.update(result)

In [ ]:
for v in completed:
  print(v.lang, v.text)